# **1. Ingesta y Auditoría del Dataset**

## **1.1 Configuración inicial**

In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## **1.2 Carga del dataset**

In [2]:
# Rutas (ajusta si tu notebook está en otra carpeta)
PROJECT_ROOT = Path("..")  # si el notebook está en book/eda/, esto apunta a book/
# Mejor: resolvemos desde el CWD actual
cwd = Path.cwd()

# Intento: detectar ruta del proyecto (sube hasta encontrar carpeta 'data')
def find_project_root(start: Path, marker: str = "data", max_up: int = 6) -> Path:
    p = start.resolve()
    for _ in range(max_up):
        if (p / marker).exists():
            return p
        p = p.parent
    return start.resolve()

ROOT = find_project_root(cwd)
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

file_path = DATA_RAW / "Ventas_1.csv"
assert file_path.exists(), f"No encuentro el archivo en: {file_path}"

# Nota: el archivo tiene separador estándar y decimales con coma en algunas columnas.
df_raw = pd.read_csv(file_path, dtype=str)  # leer como texto para controlar conversiones
print("Ruta:", file_path)
print("Shape:", df_raw.shape)
df_raw.head(5)

Ruta: C:\Users\juana\olimpica_book\data\raw\Ventas_1.csv
Shape: (345, 12)


,NroReg,FECHA,CENTRO,Estrato,OFERTA_ID,FACTURA,GRUPO_CATEG,PLU_SAP,CANTIDAD,VENTA,DESCUENTO,GRUCOM
0,45,46023,1,4,0,2,1,3,"2,061",36900,"0,00",1
1,61,46023,1,4,0,4,1,5,"7,095",47900,"0,00",1
2,63,46023,1,4,0,6,1,7,"0,984",30450,"0,00",1
3,83,46023,1,4,0,8,1,9,"0,771",17750,"0,00",1
4,88,46023,1,4,0,10,1,11,"0,402",12150,"0,00",1


##  Carga y Validación Inicial de los Datos – Ventas Olímpica

###  1. Detección dinámica de la ruta del proyecto
El código implementa una búsqueda automática de la carpeta raíz del proyecto (`data/`), lo que garantiza que el notebook sea **portátil y reproducible** en distintos entornos (local, servidor o nube).  
Esto evita errores por rutas absolutas y facilita la colaboración dentro del equipo de analítica.



###  2. Organización estructurada de los datos
Se establecen claramente dos rutas:

- `data/raw` → Datos originales sin transformación.
- `data/processed` → Datos limpios y listos para modelado.

Además, el código crea automáticamente la carpeta `processed` si no existe, asegurando una arquitectura de datos ordenada y alineada con estándares de proyectos de ciencia de datos.


###  3. Lectura controlada del archivo de ventas
El archivo `Ventas_1.csv` se carga inicialmente como texto (`dtype=str`).  
Esto es estratégico porque:

- Permite manejar correctamente columnas con **decimales usando coma**.
- Evita conversiones automáticas incorrectas.
- Da control total sobre el proceso posterior de limpieza y tipificación.



###  4. Validación inicial del dataset
La salida indica:

 **Dimensión del dataset:** `(345, 12)`  
   345 registros de transacciones con 12 variables asociadas.
  
-  Variables relevantes identificadas:
  - Fecha (`FECHA`)
  - Centro (`CENTRO`)
  - Estrato
  - Oferta (`OFERTA_ID`)
  - Categoría (`GRUPO_CATEG`)
  - Producto (`PLU_SAP`)
  - Cantidad vendida
  - Valor de venta
  - Descuento aplicado

Esto confirma que el dataset contiene tanto variables operativas como comerciales, lo que lo hace adecuado para:

- Análisis de comportamiento de ventas
- Evaluación de impacto de descuentos
- Comparación entre centros
- Modelos de predicción de demanda (series de tiempo)


###  Valor estratégico para el proyecto

Este primer paso garantiza:

- Integridad estructural del archivo
- Reproducibilidad del análisis
- Base sólida para limpieza, transformación y modelado predictivo

Desde la perspectiva gerencial, este enfoque demuestra que el proyecto no solo busca generar modelos, sino hacerlo bajo un esquema metodológico robusto y auditable.


 **Conclusión:**  
La carga de datos está correctamente estructurada y validada. El dataset presenta una base coherente para iniciar el análisis exploratorio y posteriormente avanzar hacia modelos predictivos de ventas, alineados con la estrategia analítica de Olímpica.

## **1.3 Vista rápida y estructura general**

In [3]:
print("Columnas:", list(df_raw.columns))
print("\nMuestra aleatoria:")
df_raw.sample(min(5, len(df_raw)), random_state=7)

Columnas: ['NroReg', 'FECHA', 'CENTRO', 'Estrato', 'OFERTA_ID', 'FACTURA', 'GRUPO_CATEG', 'PLU_SAP', 'CANTIDAD', 'VENTA', 'DESCUENTO', 'GRUCOM']

Muestra aleatoria:


,NroReg,FECHA,CENTRO,Estrato,OFERTA_ID,FACTURA,GRUPO_CATEG,PLU_SAP,CANTIDAD,VENTA,DESCUENTO,GRUCOM
173,3198,46023,1,4,0,158,1,254,"0,361",27700,"0,00",1
30,491,46023,1,4,0,61,1,62,"0,421",25500,"0,00",1
115,2075,46023,1,4,0,129,1,190,"0,389",29900,"0,00",1
342,6032,46023,1,4,251106,291,1,198,"0,152",4005,"445,00",1
141,2537,46023,1,4,0,218,1,219,"2,952",9570,"0,00",1




## Validación visual de columnas y muestra de datos

La salida presentada confirma que el dataset de ventas contiene 12 columnas correctamente estructuradas, cubriendo información transaccional, comercial, operativa y de producto. Esto valida que la base entregada mantiene la integridad esperada del modelo de datos.

En la muestra aleatoria observada se identifican los siguientes puntos relevantes:

- Consistencia en la estructura de los registros.
- Presencia de valores decimales en la variable `CANTIDAD`, lo que indica ventas fraccionadas o unidades no enteras.
- Existencia de registros con descuento aplicado (`DESCUENTO` distinto de cero), lo que permitirá analizar el impacto promocional en el comportamiento de ventas.
- Coherencia entre `CANTIDAD` y `VENTA`, lo que sugiere integridad inicial en los datos monetarios.

Desde una perspectiva ejecutiva, esta validación preliminar confirma que la base de datos es consistente y contiene variables estratégicas suficientes para realizar análisis de desempeño comercial y avanzar hacia modelos predictivos de demanda y optimización de ventas.


## **1.4 Diccionario de datos (tipos, nulos, cardinalidades)**

In [4]:
def data_dictionary(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "col": df.columns,
        "dtype_raw": [df[c].dtype for c in df.columns],
        "n_null": [df[c].isna().sum() for c in df.columns],
        "pct_null": [df[c].isna().mean() for c in df.columns],
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
        "sample_1": [df[c].dropna().iloc[0] if df[c].dropna().shape[0] else None for c in df.columns],
    })
    return out.sort_values(["pct_null", "n_unique"], ascending=[False, True]).reset_index(drop=True)

dic_raw = data_dictionary(df_raw)
dic_raw

,col,dtype_raw,n_null,pct_null,n_unique,sample_1
0,FECHA,str,0,0.000,1,46023
1,CENTRO,str,0,0.000,1,1
2,Estrato,str,0,0.000,1,4
3,GRUPO_CATEG,str,0,0.000,1,1
4,GRUCOM,str,0,0.000,1,1
5,OFERTA_ID,str,0,0.000,5,0
6,DESCUENTO,str,0,0.000,24,"0,00"
7,FACTURA,str,0,0.000,175,2
8,VENTA,str,0,0.000,195,36900
9,CANTIDAD,str,0,0.000,210,"2,061"



##  Diccionario de Datos – Evaluación Integral de Calidad

El bloque presentado construye un diccionario técnico del dataset con el objetivo de auditar la calidad estructural de la información antes de iniciar procesos de transformación y modelado.

###  1. Control de calidad estructural

La salida muestra que:

- Todas las columnas tienen `n_null = 0`
- El porcentaje de valores nulos (`pct_null`) es `0.000` en todos los casos

Esto indica **integridad total en la captura de datos** dentro de esta muestra. Desde una perspectiva operativa, refleja consistencia en los sistemas transaccionales que originan la información.



###  2. Tipificación actual de las variables

Se observa que todas las variables están cargadas como tipo `str`.

Esta decisión fue intencional en la etapa inicial para evitar errores en:

- Valores monetarios (`VENTA`)
- Cantidades con separador decimal tipo coma (`CANTIDAD`)
- Descuentos (`DESCUENTO`)

Sin embargo, para análisis cuantitativo y modelos predictivos será necesario:

- Convertir `VENTA`, `CANTIDAD` y `DESCUENTO` a formato numérico
- Transformar `FECHA` a tipo fecha
- Definir variables categóricas (por ejemplo `CENTRO`, `GRUPO_CATEG`, `Estrato`)


###  3. Análisis de cardinalidad (valores únicos)

El número de valores únicos permite entender la naturaleza de cada variable:

- `NroReg` presenta 345 valores únicos → identificador por registro
- `PLU_SAP` presenta 228 valores únicos → alta diversidad de productos
- `FACTURA` presenta 175 valores únicos → múltiples productos por transacción
- `DESCUENTO` presenta 24 valores únicos → posible análisis de políticas promocionales
- `CENTRO`, `Estrato`, `GRUPO_CATEG` presentan baja cardinalidad → ideales para segmentación

Este análisis es estratégico porque permite anticipar:

- Variables explicativas relevantes para modelos de predicción
- Potencial de segmentación comercial
- Nivel de granularidad disponible para análisis de desempeño


###  Implicaciones estratégicas

La revisión confirma que la base:

- Tiene alta calidad estructural
- No presenta valores faltantes
- Contiene diversidad suficiente para análisis por producto, centro y factura
- Está lista para una fase de tipificación y transformación controlada


Conclusión  
El diccionario de datos confirma que la información entregada cumple con estándares adecuados de integridad y diversidad analítica, lo que permite avanzar con confianza hacia análisis descriptivos avanzados y modelos predictivos orientados a optimización comercial.


## **1.5 Conversión de tipos críticos**

In [5]:
df = df_raw.copy()

# Helpers robustos
def to_numeric_comma(series: pd.Series) -> pd.Series:
    """Convierte strings con coma decimal y separadores raros a float (NaN si falla)."""
    s = series.astype(str).str.strip()
    s = s.replace({"None": np.nan, "nan": np.nan, "": np.nan})
    s = s.str.replace(".", "", regex=False)      # por si hay miles con punto (poco probable)
    s = s.str.replace(",", ".", regex=False)     # coma decimal -> punto
    return pd.to_numeric(s, errors="coerce")

def excel_serial_to_date(series: pd.Series) -> pd.Series:
    """Convierte serial excel (días desde 1899-12-30) a datetime."""
    s = pd.to_numeric(series, errors="coerce")
    return pd.to_datetime("1899-12-30") + pd.to_timedelta(s, unit="D")

# Ajusta nombres si cambian
cols_expected = ["NroReg", "FECHA", "FACTURA", "PLU_SAP", "CENTRO", "VENTA", "CANTIDAD", "DESCUENTO", "OFERTA_ID"]
missing = [c for c in cols_expected if c not in df.columns]
print("Faltan (si aplica):", missing)

# Conversión
if "NroReg" in df.columns:
    df["NroReg"] = pd.to_numeric(df["NroReg"], errors="coerce").astype("Int64")

if "FECHA" in df.columns:
    df["FECHA"] = excel_serial_to_date(df["FECHA"])

for c in ["VENTA", "CANTIDAD", "DESCUENTO"]:
    if c in df.columns:
        df[c] = to_numeric_comma(df[c])

for c in ["FACTURA", "PLU_SAP", "CENTRO", "GRUPO_CATEG", "GRUCOM", "Estrato", "OFERTA_ID"]:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

# Tipos finales
df.dtypes

Faltan (si aplica): []


NroReg                  Int64
FECHA          datetime64[us]
CENTRO                    str
Estrato                   str
OFERTA_ID                 str
FACTURA                   str
GRUPO_CATEG               str
PLU_SAP                   str
CANTIDAD              float64
VENTA                   int64
DESCUENTO             float64
GRUCOM                    str
dtype: object


## Estandarización de tipos y preparación del dataset para analítica

La salida confirma que el proceso de tipificación y limpieza fue exitoso y deja el dataset listo para análisis exploratorio cuantitativo y, posteriormente, para modelado predictivo.

### Validación de columnas esperadas

La línea `Faltan (si aplica): []` indica que todas las columnas críticas definidas como necesarias para el análisis (`FECHA`, `VENTA`, `CANTIDAD`, `DESCUENTO`, identificadores y llaves operativas) están presentes. Esto reduce riesgos de reprocesos y garantiza que el flujo de trabajo puede continuar sin ajustes de esquema.

### Conversión de fecha a formato analítico

`FECHA` aparece como `datetime64[us]`, lo que confirma que la variable fue convertida correctamente desde un serial tipo Excel a una fecha real. Este paso es fundamental porque habilita análisis temporales como:

- tendencias por día/semana/mes
- estacionalidad
- construcción de series de tiempo para predicción de ventas

### Variables numéricas correctamente tipificadas

Las variables clave para el análisis monetario y de volumen quedan como numéricas:

- `VENTA` → `int64`
- `CANTIDAD` → `float64`
- `DESCUENTO` → `float64`

Esto confirma que se resolvió adecuadamente el reto de lectura inicial (valores con coma decimal) y que ahora es posible:

- calcular métricas agregadas (sumas, promedios, percentiles)
- evaluar elasticidad a descuentos
- construir variables derivadas (precio unitario, ticket promedio, etc.)

### Identificadores y dimensiones como texto

Dimensiones como `CENTRO`, `PLU_SAP`, `GRUPO_CATEG`, `OFERTA_ID`, `Estrato` y `GRUCOM` permanecen como `str`, lo cual es adecuado porque se comportan como variables categóricas y son esenciales para segmentación y análisis comparativo.

### Robustez del proceso de limpieza

Los resultados reflejan que el pipeline implementa controles prácticos de calidad:

- normalización de separadores decimales y posibles separadores de miles
- conversión segura con `errors="coerce"` para evitar fallas en ejecución
- estandarización de texto con `strip()` para eliminar espacios e inconsistencias

## Conclusión

La tipificación final confirma que el dataset ya está en un estado “listo para analítica”: fechas interpretables, variables numéricas operables y dimensiones categóricas consistentes. Esto habilita una fase de exploración más profunda y crea una base sólida para modelos de pronóstico de ventas alineados con los objetivos del proyecto.


## **1.6 Validación de granularidad y llaves (factura-producto-fecha-centro)**

In [6]:
# Granularidad esperada: una línea de factura (producto dentro de factura)
# Posible llave: FECHA + CENTRO + FACTURA + PLU_SAP
key_cols = [c for c in ["FECHA", "CENTRO", "FACTURA", "PLU_SAP"] if c in df.columns]

print("Key cols:", key_cols)
if len(key_cols) == 4:
    dup_key = df.duplicated(subset=key_cols).sum()
    print("Duplicados por llave (FECHA-CENTRO-FACTURA-PLU_SAP):", dup_key)
else:
    print("No están todas las columnas para la llave completa.")

# Conteos de cardinalidad útiles para auditoría
summary = {
    "n_rows": len(df),
    "n_cols": df.shape[1],
    "n_dates": df["FECHA"].nunique() if "FECHA" in df.columns else None,
    "n_centros": df["CENTRO"].nunique() if "CENTRO" in df.columns else None,
    "n_facturas": df["FACTURA"].nunique() if "FACTURA" in df.columns else None,
    "n_products": df["PLU_SAP"].nunique() if "PLU_SAP" in df.columns else None,
}
summary

Key cols: ['FECHA', 'CENTRO', 'FACTURA', 'PLU_SAP']
Duplicados por llave (FECHA-CENTRO-FACTURA-PLU_SAP): 0


{'n_rows': 345,
 'n_cols': 12,
 'n_dates': 1,
 'n_centros': 1,
 'n_facturas': 175,
 'n_products': 228}


## Auditoría de granularidad y unicidad de registros

La salida valida un aspecto crítico del dataset: que la unidad de análisis corresponde a una “línea de factura” (producto dentro de una factura), y que dicha granularidad puede identificarse de forma consistente mediante una llave compuesta.

### Llave operacional utilizada

Se definieron como columnas clave:

- `FECHA`
- `CENTRO`
- `FACTURA`
- `PLU_SAP`

Esta combinación representa, de forma natural, una transacción a nivel de producto dentro de una factura, en un centro específico y en una fecha determinada.

### Resultado de duplicados por llave

La auditoría arroja:

- `Duplicados por llave (FECHA-CENTRO-FACTURA-PLU_SAP): 0`

Esto es un indicador fuerte de calidad, porque confirma que no existen registros repetidos para la misma combinación clave. En términos prácticos, cada fila del dataset es única bajo la granularidad esperada, lo que evita distorsiones en métricas como ventas totales, volumen, ticket promedio o análisis de promociones.

### Resumen de cardinalidades relevantes

Los conteos reportados permiten entender rápidamente la estructura del dataset:

- `n_rows: 345` registros (líneas de venta)
- `n_cols: 12` variables disponibles
- `n_dates: 1` fecha única en la muestra analizada
- `n_centros: 1` centro único en la muestra analizada
- `n_facturas: 175` facturas distintas
- `n_products: 228` productos distintos

Interpretación operativa:

- La muestra representa un solo día y un solo centro, lo cual es consistente con un extracto inicial o una partición del histórico.
- Existe una relación de múltiples productos por factura: 175 facturas para 345 líneas, lo que sugiere aproximadamente 2 ítems por factura en promedio en esta muestra.
- La diversidad de productos (228) indica buena variedad de referencias, incluso en un extracto de tamaño limitado.

## Conclusión

Esta auditoría confirma que el dataset está correctamente estructurado a nivel de “línea de factura” y que no presenta duplicidades bajo la llave operacional definida. Esto garantiza que los análisis posteriores (agregaciones, segmentación por producto o factura, y construcción de series de tiempo) se basarán en una representación consistente y confiable de las transacciones.


## **1.7 Auditoría de calidad (nulos, duplicados, rangos, valores inválidos)**

In [7]:
audit = {}

# Nulos
audit["nulls"] = df.isna().sum().to_dict()

# Duplicados fila completa
audit["duplicate_rows_full"] = int(df.duplicated().sum())

# Rango básico
for c in ["VENTA", "CANTIDAD", "DESCUENTO"]:
    if c in df.columns:
        audit[f"{c}_min"] = float(np.nanmin(df[c].values))
        audit[f"{c}_max"] = float(np.nanmax(df[c].values))
        audit[f"{c}_neg_count"] = int((df[c] < 0).sum())

# Fechas raras
if "FECHA" in df.columns:
    audit["fecha_min"] = str(df["FECHA"].min())
    audit["fecha_max"] = str(df["FECHA"].max())
    audit["fecha_na"] = int(df["FECHA"].isna().sum())

audit

{'nulls': {'NroReg': 0,
  'FECHA': 0,
  'CENTRO': 0,
  'Estrato': 0,
  'OFERTA_ID': 0,
  'FACTURA': 0,
  'GRUPO_CATEG': 0,
  'PLU_SAP': 0,
  'CANTIDAD': 0,
  'VENTA': 0,
  'DESCUENTO': 0,
  'GRUCOM': 0},
 'duplicate_rows_full': 0,
 'VENTA_min': 1200.0,
 'VENTA_max': 77790.0,
 'VENTA_neg_count': 0,
 'CANTIDAD_min': 0.088,
 'CANTIDAD_max': 9.906,
 'CANTIDAD_neg_count': 0,
 'DESCUENTO_min': 0.0,
 'DESCUENTO_max': 13510.0,
 'DESCUENTO_neg_count': 0,
 'fecha_min': '2026-01-01 00:00:00',
 'fecha_max': '2026-01-01 00:00:00',
 'fecha_na': 0}


## Auditoría de integridad y validación de rangos operativos

La salida presentada corresponde a una auditoría estructural y numérica del dataset, orientada a validar consistencia, integridad y posibles anomalías antes de avanzar hacia análisis estratégicos o modelado predictivo.

### 1. Control de valores nulos

El bloque reporta:

- Todos los campos con 0 valores nulos.
- `fecha_na: 0`

Esto confirma integridad total en la muestra analizada. No se requieren procesos de imputación ni ajustes correctivos en esta etapa.

### 2. Control de duplicados completos

- `duplicate_rows_full: 0`

No existen filas completamente duplicadas en el dataset. Esto reduce el riesgo de sobreestimación en métricas como ventas totales, volumen vendido o número de transacciones.

### 3. Validación de rangos en variables críticas

#### VENTA
- Mínimo: 1.200
- Máximo: 77.790
- Registros negativos: 0

Los valores monetarios son coherentes y no presentan ventas negativas, lo cual es consistente con operaciones normales de facturación.

#### CANTIDAD
- Mínimo: 0.088
- Máximo: 9.906
- Registros negativos: 0

Se observan cantidades fraccionadas, lo que sugiere venta por peso o unidades parciales. No existen valores negativos, lo cual valida consistencia operativa.

#### DESCUENTO
- Mínimo: 0.0
- Máximo: 13.510
- Registros negativos: 0

El rango indica presencia de descuentos significativos en algunos registros, lo cual será relevante para analizar impacto promocional y elasticidad en ventas.

### 4. Consistencia temporal

- `fecha_min`: 2026-01-01
- `fecha_max`: 2026-01-01

La muestra corresponde a un único día, lo cual es consistente con un extracto puntual del histórico. Esto debe considerarse al interpretar métricas agregadas y antes de construir modelos de series de tiempo.

## Conclusión ejecutiva

La auditoría confirma que el dataset:

- No contiene valores faltantes.
- No presenta duplicados completos.
- No tiene valores negativos en variables monetarias ni de volumen.
- Mantiene rangos coherentes para ventas, cantidades y descuentos.
- Está acotado a una única fecha en esta muestra.

En términos de calidad de datos, la base se encuentra en condiciones adecuadas para análisis exploratorio avanzado y posterior modelado predictivo, sin riesgos estructurales evidentes en esta etapa.


## **1.8 Reglas de negocio: ventas netas y señal de promoción**

In [8]:
# Ventas netas (asumiendo VENTA = bruto y DESCUENTO = descuento en moneda)
if {"VENTA", "DESCUENTO"}.issubset(df.columns):
    df["VENTA_NETA"] = df["VENTA"] - df["DESCUENTO"]
else:
    df["VENTA_NETA"] = np.nan

# promo_flag a partir de OFERTA_ID o descuento
if "OFERTA_ID" in df.columns:
    df["OFERTA_ID_NUM"] = pd.to_numeric(df["OFERTA_ID"], errors="coerce").fillna(0).astype(int)
    df["PROMO_FLAG"] = (df["OFERTA_ID_NUM"] > 0).astype(int)
else:
    df["PROMO_FLAG"] = (df.get("DESCUENTO", 0).fillna(0) > 0).astype(int)

# precio unitario neto (solo exploratorio; cuidado si cantidad es por peso)
if {"VENTA_NETA", "CANTIDAD"}.issubset(df.columns):
    df["PRECIO_UNITARIO_NETO"] = np.where(df["CANTIDAD"] > 0, df["VENTA_NETA"] / df["CANTIDAD"], np.nan)

df[["VENTA", "DESCUENTO", "VENTA_NETA", "CANTIDAD", "PROMO_FLAG", "PRECIO_UNITARIO_NETO"]].head(10)

,VENTA,DESCUENTO,VENTA_NETA,CANTIDAD,PROMO_FLAG,PRECIO_UNITARIO_NETO
0,36900,0.000,"36,900.000",2.061,0,"17,903.930"
1,47900,0.000,"47,900.000",7.095,0,"6,751.233"
2,30450,0.000,"30,450.000",0.984,0,"30,945.122"
3,17750,0.000,"17,750.000",0.771,0,"23,022.049"
4,12150,0.000,"12,150.000",0.402,0,"30,223.881"
5,1200,0.000,"1,200.000",0.321,0,"3,738.318"
6,17980,0.000,"17,980.000",0.792,0,"22,702.020"
7,11850,0.000,"11,850.000",0.633,0,"18,720.379"
8,31400,0.000,"31,400.000",5.402,0,"5,812.662"
9,31800,0.000,"31,800.000",0.580,0,"54,827.586"



## Derivación de variables de negocio para análisis y modelado

La salida evidencia la creación de tres variables derivadas clave que enriquecen el dataset y lo alinean con preguntas típicas de negocio: ventas netas, presencia de promoción y precio unitario neto.

### VENTA_NETA: medición real del ingreso por línea

A partir de `VENTA` y `DESCUENTO` se calcula:

- `VENTA_NETA = VENTA - DESCUENTO`

En la muestra mostrada, la mayoría de registros tienen `DESCUENTO = 0`, por lo que `VENTA_NETA` coincide con `VENTA`. Este resultado es útil para:

- consolidar ingresos netos por producto, factura o centro
- comparar desempeño entre promociones vs no promociones
- construir series de tiempo con una métrica homogénea para pronóstico

### PROMO_FLAG: identificación operativa de promoción

La salida incluye una bandera binaria:

- `PROMO_FLAG = 1` si `OFERTA_ID > 0`  
- `PROMO_FLAG = 0` en caso contrario

En los 10 registros presentados, `PROMO_FLAG = 0`, lo que es consistente con `DESCUENTO = 0` en esa muestra. Esta variable permite análisis directos como:

- participación de ventas en promoción vs no promoción
- elasticidad de demanda frente a promociones
- evaluación de efectividad promocional por categoría o producto

### PRECIO_UNITARIO_NETO: señal de precio implícito

Se calcula como:

- `PRECIO_UNITARIO_NETO = VENTA_NETA / CANTIDAD` (cuando `CANTIDAD > 0`)

En la salida se observan precios unitarios que varían ampliamente entre filas, lo cual es esperable porque:

- `CANTIDAD` puede representar unidades fraccionadas (por ejemplo, productos vendidos por peso)
- se trata de diferentes productos (`PLU_SAP`) con estructuras de precio distintas

Este indicador es particularmente valioso para:

- detectar outliers de precio (posibles errores de captura o casos atípicos)
- construir features para modelos (precio unitario, dispersión por producto)
- estudiar sensibilidad del volumen ante cambios en precio neto

## Conclusión

La tabla resultante confirma que el dataset ya incorpora variables interpretables desde negocio. Estas transformaciones habilitan análisis más directos y accionables, y mejoran la capacidad explicativa para modelos predictivos de ventas, especialmente al distinguir ventas netas, actividad promocional y señales de precio unitario.


## **1.9 Estadísticas descriptivas (ventas, cantidades, descuentos)**

In [9]:
def describe_numeric(cols):
    cols = [c for c in cols if c in df.columns]
    return df[cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

describe_numeric(["VENTA", "DESCUENTO", "VENTA_NETA", "CANTIDAD", "PRECIO_UNITARIO_NETO"])

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
VENTA,345.000,"16,541.858","11,738.521","1,200.000","1,200.000","3,800.000","8,250.000","13,650.000","22,725.000","38,860.000","52,808.000","77,790.000"
DESCUENTO,345.000,247.272,"1,250.130",0.000,0.000,0.000,0.000,0.000,0.000,"1,463.800","5,806.000","13,510.000"
VENTA_NETA,345.000,"16,294.586","11,540.719","1,200.000","1,200.000","3,566.400","8,250.000","13,220.000","21,990.000","38,320.000","50,760.400","77,790.000"
CANTIDAD,345.000,0.868,1.195,0.088,0.128,0.156,0.321,0.518,0.962,2.428,7.095,9.906
PRECIO_UNITARIO_NETO,345.000,"32,372.717","31,720.941","2,944.515","3,241.870","4,455.899","10,991.189","23,421.053","41,610.738","85,531.429","158,602.500","248,458.150"



## Estadísticos descriptivos de variables críticas de negocio

La tabla presentada resume la distribución de las principales variables cuantitativas del dataset, incorporando percentiles avanzados (1%, 5%, 95% y 99%) para evaluar dispersión y posibles valores extremos.

### 1. VENTA y VENTA_NETA

- Total de registros: 345
- Venta promedio: 16.542
- Mediana (P50): 13.650
- P95: 52.808
- Máximo: 77.790

La media es superior a la mediana, lo que sugiere una distribución asimétrica positiva, con presencia de tickets altos que elevan el promedio.  
Este comportamiento es consistente con ventas minoristas donde existen algunos productos de mayor valor o compras voluminosas.

La similitud entre `VENTA` y `VENTA_NETA` indica que, en esta muestra, el impacto promedio de descuentos es relativamente bajo.

### 2. DESCUENTO

- Media: 247
- Mediana: 0
- P75: 0
- P95: 1.463
- Máximo: 13.510

El hecho de que la mediana y el P75 sean cero confirma que la mayoría de transacciones no tienen descuento aplicado.  
Sin embargo, el máximo elevado evidencia que existen eventos promocionales puntuales de alto impacto, lo cual será relevante para analizar elasticidad y efecto promocional.

### 3. CANTIDAD

- Media: 0.868
- Mediana: 0.518
- P95: 2.428
- Máximo: 9.906

La media superior a la mediana también sugiere asimetría positiva.  
La presencia de valores fraccionados confirma que existen productos vendidos por peso o fracción de unidad.  
Algunos registros muestran compras de alto volumen (cercanos a 10 unidades o kilogramos), que pueden influir significativamente en ingresos totales.

### 4. PRECIO_UNITARIO_NETO

- Media: 32.373
- Mediana: 23.421
- P95: 158.603
- Máximo: 248.458

La dispersión es alta, lo cual es esperable dado que se trata de múltiples productos con estructuras de precio distintas.  
La diferencia significativa entre mediana y P95 indica la presencia de productos premium o de alto valor unitario.

## Lectura ejecutiva

Los estadísticos descriptivos muestran:

- Distribuciones asimétricas en ventas y cantidades.
- Bajo uso generalizado de descuentos, con eventos promocionales concentrados.
- Alta heterogeneidad en precios unitarios.

Desde una perspectiva estratégica, estos resultados respaldan la necesidad de:

- Analizar comportamiento por segmento de producto.
- Evaluar impacto específico de promociones en productos de alto ticket.
- Considerar transformaciones (por ejemplo, logaritmos) en modelos predictivos debido a la asimetría observada.




## **1.10 Outliers y valores extremos (diagnóstico inicial)**

In [10]:
# Top 10 por venta neta
if "VENTA_NETA" in df.columns:
    top_venta = df.sort_values("VENTA_NETA", ascending=False).head(10)
    top_venta[["FECHA", "CENTRO", "FACTURA", "PLU_SAP", "CANTIDAD", "VENTA", "DESCUENTO", "VENTA_NETA", "PROMO_FLAG"]]

# Top 10 por descuento
if "DESCUENTO" in df.columns:
    top_desc = df.sort_values("DESCUENTO", ascending=False).head(10)
    top_desc[["FECHA", "CENTRO", "FACTURA", "PLU_SAP", "CANTIDAD", "VENTA", "DESCUENTO", "VENTA_NETA", "PROMO_FLAG"]]

## **1.11 Checks de consistencia (oferta vs descuento, signos, coherencia básica)**

In [11]:
checks = {}

# 1) Venta neta negativa (raro si no hay devoluciones)
if "VENTA_NETA" in df.columns:
    checks["venta_neta_negativa"] = int((df["VENTA_NETA"] < 0).sum())

# 2) Descuento > venta (raro)
if {"VENTA", "DESCUENTO"}.issubset(df.columns):
    checks["descuento_mayor_que_venta"] = int((df["DESCUENTO"] > df["VENTA"]).sum())

# 3) promo_flag vs descuento (consistencia)
if {"PROMO_FLAG", "DESCUENTO"}.issubset(df.columns):
    checks["promo_flag_1_descuento_0"] = int(((df["PROMO_FLAG"] == 1) & (df["DESCUENTO"] == 0)).sum())
    checks["promo_flag_0_descuento_gt_0"] = int(((df["PROMO_FLAG"] == 0) & (df["DESCUENTO"] > 0)).sum())

# 4) Cantidad <= 0 (raro)
if "CANTIDAD" in df.columns:
    checks["cantidad_le_0"] = int((df["CANTIDAD"] <= 0).sum())

checks

{'venta_neta_negativa': 0,
 'descuento_mayor_que_venta': 0,
 'promo_flag_1_descuento_0': 0,
 'promo_flag_0_descuento_gt_0': 0,
 'cantidad_le_0': 0}


## Controles de consistencia bajo reglas de negocio

El bloque ejecutado valida coherencia lógica entre variables clave del dataset, aplicando reglas básicas de negocio para identificar posibles anomalías operativas.

### Resultados de validación

- `venta_neta_negativa: 0`
- `descuento_mayor_que_venta: 0`
- `promo_flag_1_descuento_0: 0`
- `promo_flag_0_descuento_gt_0: 0`
- `cantidad_le_0: 0`

### Interpretación ejecutiva

**1. Venta neta negativa = 0**  
No existen registros donde la venta neta sea negativa. Esto indica ausencia de devoluciones o errores de captura en esta muestra, y garantiza coherencia en el cálculo de ingresos.

**2. Descuento mayor que la venta = 0**  
No se detectan casos donde el descuento supere el valor bruto de la venta. Esta validación es crítica, ya que evitaría distorsiones en métricas financieras y márgenes.

**3. Consistencia entre promoción y descuento**  
No se encontraron inconsistencias entre `PROMO_FLAG` y `DESCUENTO`.  
Esto significa que:
- No existen promociones marcadas sin descuento aplicado.
- No existen descuentos aplicados sin una bandera promocional asociada.

Este resultado refuerza la confiabilidad del modelo de datos promocional.

**4. Cantidad menor o igual a cero = 0**  
Todas las cantidades vendidas son positivas. No hay registros con valores cero o negativos, lo que confirma integridad transaccional.

## Conclusión

Las validaciones muestran que el dataset cumple con reglas básicas de coherencia financiera y operativa. No se detectan anomalías estructurales ni inconsistencias lógicas en esta muestra.

Desde una perspectiva analítica, esto reduce significativamente el riesgo de sesgos en métricas agregadas y fortalece la confiabilidad del dataset como insumo para análisis exploratorio avanzado y modelos predictivos.


## **1.12 Guardado de dataset limpio (processed) y log de auditoría**

In [12]:
# Guardar dataset limpio (muestra) en processed
out_csv = DATA_PROCESSED / "Ventas_1_clean.csv"
df.to_csv(out_csv, index=False, encoding="utf-8")
print("Guardado:", out_csv)

# Log de auditoría (json)
audit_report = {
    "file": str(file_path),
    "shape_raw": list(df_raw.shape),
    "shape_clean": list(df.shape),
    "summary": summary,
    "checks": checks,
    "nulls": audit["nulls"],
    "duplicate_rows_full": audit["duplicate_rows_full"],
}
out_json = DATA_PROCESSED / "audit_Ventas_1.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(audit_report, f, ensure_ascii=False, indent=2)

print("Audit report:", out_json)
audit_report

Guardado: C:\Users\juana\olimpica_book\data\processed\Ventas_1_clean.csv
Audit report: C:\Users\juana\olimpica_book\data\processed\audit_Ventas_1.json


{'file': 'C:\\Users\\juana\\olimpica_book\\data\\raw\\Ventas_1.csv',
 'shape_raw': [345, 12],
 'shape_clean': [345, 16],
 'summary': {'n_rows': 345,
  'n_cols': 12,
  'n_dates': 1,
  'n_centros': 1,
  'n_facturas': 175,
  'n_products': 228},
 'checks': {'venta_neta_negativa': 0,
  'descuento_mayor_que_venta': 0,
  'promo_flag_1_descuento_0': 0,
  'promo_flag_0_descuento_gt_0': 0,
  'cantidad_le_0': 0},
 'nulls': {'NroReg': 0,
  'FECHA': 0,
  'CENTRO': 0,
  'Estrato': 0,
  'OFERTA_ID': 0,
  'FACTURA': 0,
  'GRUPO_CATEG': 0,
  'PLU_SAP': 0,
  'CANTIDAD': 0,
  'VENTA': 0,
  'DESCUENTO': 0,
  'GRUCOM': 0},
 'duplicate_rows_full': 0}


## Persistencia del dataset limpio y trazabilidad del proceso

El bloque ejecutado cumple una función estratégica dentro del proyecto: no solo transforma y valida la información, sino que garantiza su trazabilidad y gobernanza mediante la generación de artefactos formales.

### 1. Generación del dataset limpio

Se guarda el archivo:

- `Ventas_1_clean.csv` en la carpeta `data/processed`

Esto permite:

- Separar claramente datos originales (`raw`) de datos transformados (`processed`)
- Garantizar reproducibilidad del análisis
- Facilitar auditorías posteriores
- Evitar reprocesamientos innecesarios

El cambio estructural evidencia valor agregado:

- Shape original: 345 x 12
- Shape limpio: 345 x 16

Las 4 columnas adicionales corresponden a variables derivadas estratégicas (por ejemplo, venta neta, bandera promocional y precio unitario neto), que enriquecen el análisis sin alterar la integridad transaccional.


### 2. Generación del reporte de auditoría (JSON)

Se crea el archivo:

- `audit_Ventas_1.json`

Este reporte consolida:

- Ruta del archivo fuente
- Dimensiones antes y después del procesamiento
- Resumen de cardinalidades
- Validaciones de reglas de negocio
- Conteo de nulos
- Duplicados completos



### 3. Hallazgos clave consolidados en el reporte

- No existen valores nulos.
- No hay duplicados completos.
- No se detectan inconsistencias financieras (ventas negativas, descuentos superiores a la venta, etc.).
- La muestra corresponde a 1 fecha y 1 centro.
- Se observan 175 facturas y 228 productos distintos.


